In [ ]:
create database if not exists funnel_db; -----database , schemea and table creation----

use database funnel_db;

create schema if not exists raw;
create schema if not exists processed;
create schema if not exists analysis;

create or replace table raw.events_raw(
event_time timestamp_tz,
event_type string,
product_id int,
category_id bigint,
category_code string,
brand string,
price int,
user_id int,
user_session varchar
)


In [ ]:
drop table if exists raw.events_raw_stage;

create or replace table funnel_db.raw.events_raw_stage(
event_time string,
event_type string,
product_id int,
category_id bigint,
category_code string,
brand string,
price int,
user_id int,
user_session varchar
);

use database funnel_db;
copy into raw.events_raw_stage
from '@"FUNNEL_DB"."RAW"."FUNNELPROJECT_DATA"/2019-oct.parquet'
file_format = (type=parquet)
match_by_column_name = case_insensitive
on_error = 'continue';



In [ ]:
use database funnel_db; 

copy into raw.events_raw_stage
from '@"FUNNEL_DB"."RAW"."FUNNELPROJECT_DATA"/2019-Nov.parquet'
file_format = (type=parquet)
match_by_column_name = case_insensitive
on_error = 'continue';

copy into raw.events_raw_stage
from '@"FUNNEL_DB"."RAW"."FUNNELPROJECT_DATA"/2019-Dec.parquet'
file_format = (type=parquet)
match_by_column_name = case_insensitive
on_error = 'continue';

copy into raw.events_raw_stage
from '@"FUNNEL_DB"."RAW"."FUNNELPROJECT_DATA"/2020-Jan.parquet'
file_format = (type=parquet)
match_by_column_name = case_insensitive
on_error = 'continue';

copy into raw.events_raw_stage
from '@"FUNNEL_DB"."RAW"."FUNNELPROJECT_DATA"/2020-Feb.parquet'
file_format = (type=parquet)
match_by_column_name = case_insensitive
on_error = 'continue';




In [ ]:
describe table raw.events_raw_stage;

In [ ]:
alter session set timezone = 'UTC' ;

truncate table raw.events_raw;

insert into raw.events_raw
select 
    try_to_timestamp_tz(replace(event_time,' UTC','+00:00')) as event_time,
    event_type, product_id, category_id, category_code, brand,
    price, user_id, user_session
from raw.events_raw_stage;

describe table raw.events_raw

In [ ]:
select count(*)  -- just to check the data completeness
from raw.events_raw;

In [ ]:
describe table raw.events_raw



In [ ]:
/*alter table raw.events_raw          
drop column category_code, brand;*/  ---had 90% and 43% nulls respectively + no arithmetic value-----

create or replace table raw.events_cleaned as

with dedup as (                      ------to remove the duplicates--------   
        select * , 
        row_number() over (partition by user_id, event_type, user_session,
         product_id order by event_time) rn
from raw.events_raw )

select d.user_id, d.user_session, d.event_time, d.event_type, d.product_id, d.category_id,
       abs(try_to_number(d.price)) price,date(d.event_time) event_date, r.saw_view, r.saw_cart, r.saw_removeCart, r.saw_purchased
from dedup d left join (
                        select user_id, user_session, product_id,
                        max(case when event_type = 'view' then 1 else 0 end) saw_view,
                        max(case when event_type = 'cart' then 1 else 0 end) saw_cart,
                        max(case when event_type = 'remove_from_cart' then 1 else 0 end) saw_removeCart,
                        max(case when event_type = 'purchase' then 1 else 0 end) saw_purchased
                    from dedup 
                    where rn = 1
                    group by user_id, user_session, product_id ) r
 on d.user_session = r.user_session
 and d.user_id = r.user_id
 and d.product_id = r.product_id
where d.rn = 1;



In [ ]:
select count(*)
from raw.events_cleaned;

In [ ]:
/*with dedup as (                     
        select * , 
        row_number() over (partition by user_id, event_type, user_session,
         product_id order by event_time) rn
from raw.events_raw )
select * from 
dedup
where rn >1*/

with dedup as (                     
        select * , 
        row_number() over (partition by user_id, event_type, user_session,
         product_id order by event_time) rn
from raw.events_raw )
select * from 
dedup
where category_id = 1487580006317032337
and user_id = 561233518

In [ ]:
select * from 
raw.events_cleaned
where saw_cart = 1 and saw_view = 1 
limit 20

In [ ]:
------for per product granularity as a mother table for further lower granularity table--------
create
or replace table processed.events_processed as 

with enriching as (
    select
        *,
        min(
            case
                when event_type = 'view' then event_time
            end
        ) over(partition by user_id, user_session, product_id) view_time,
        min(
            case
                when event_type = 'cart' then event_time
            end
        ) over (partition by user_id, user_session, product_id) cart_time,
        min(
            case
                when event_type = 'remove_from_cart' then event_time
            end
        ) over (partition by user_id, user_session, product_id) removeCart_time,
        min(
            case
                when event_type = 'purchase' then event_time
            end
        ) over (partition by user_id, user_session, product_id) purchase_time
    from
        raw.events_cleaned
)
select
    user_id,
    user_session,
    event_time,
    event_type,
    product_id,
    category_id,
    price,
    event_date,
    saw_view,
    saw_cart,
    saw_removeCart,
    saw_purchased,
    case
        when view_time is not null
        and cart_time is not null
        and cart_time > view_time then datediff('seconds', view_time, cart_time)
    end sec_view_toCart,
    case
        when view_time is not null
        and purchase_time is not null
        and purchase_time > view_time then datediff('seconds', view_time, purchase_time)
    end sec_view_toPurchase,
    case
        when cart_time is not null
        and purchase_time is not null
        and purchase_time > cart_time then datediff('seconds', cart_time, purchase_time)
    end sec_cart_toPurchase,
    case
        when cart_time is not null
        and removeCart_time is not null
        and removeCart_time > cart_time then datediff('seconds', cart_time, removeCart_time)
    end sec_cart_toRemoveCart
from
    enriching ;

In [ ]:
select * from 
processed.events_processed
limit 20

In [ ]:
select
    *
from
    processed.events_processed
where
    sec_view_tocart > 0

In [ ]:

------creating a table in the analysis schema for product level analysis , this has the highest granularity---------


create or replace table analysis.events_prodLevel as 

with enriching as (
    select
        *,
        min(
            case
                when event_type = 'view' then event_time
            end
        ) over(partition by user_id, user_session, product_id) view_time,
        min(
            case
                when event_type = 'cart' then event_time
            end
        ) over (partition by user_id, user_session, product_id) cart_time,
        min(
            case
                when event_type = 'remove_from_cart' then event_time
            end
        ) over (partition by user_id, user_session, product_id) removeCart_time,
        min(
            case
                when event_type = 'purchase' then event_time
            end
        ) over (partition by user_id, user_session, product_id) purchase_time
    from
        raw.events_cleaned
)
select
    user_id,
    user_session,
    event_time,
    event_type,
    product_id,
    category_id,
    price,
    event_date,
    saw_view,
    saw_cart,
    saw_removeCart,
    saw_purchased,
    view_time,
    cart_time,
    removeCart_time,
    purchase_time,
    case
        when view_time is not null
        and cart_time is not null
        and cart_time > view_time then datediff('seconds', view_time, cart_time)
    end sec_view_toCart,
    case
        when view_time is not null
        and purchase_time is not null
        and purchase_time > view_time then datediff('seconds', view_time, purchase_time)
    end sec_view_toPurchase,
    case
        when cart_time is not null
        and purchase_time is not null
        and purchase_time > cart_time then datediff('seconds', cart_time, purchase_time)
    end sec_cart_toPurchase,
    case
        when cart_time is not null
        and removeCart_time is not null
        and removeCart_time > cart_time then datediff('seconds', cart_time, removeCart_time)
    end sec_cart_toRemoveCart
from
    enriching ;




In [ ]:
select *
from analysis.events_prodLevel
limit 20

In [ ]:
-------------creating a table with session level granularity can have multiple user_id but no product repetion ------------

create or replace table analysis.events_sessionLevel as 

select 
        user_id,
        user_session,
        min(event_date) session_date,
        min(event_time) first_event_time,
        max(event_time) last_event_time,

        max(saw_view) saw_view,
        max(saw_cart) saw_cart,
        max(saw_purchased) saw_purchase,
        max(saw_removecart) saw_removeCart,

        min(view_time) first_view_time,
        min(cart_time) first_cart_time,
        min(removeCart_time) first_removeCart_time,
        min(purchase_time) first_purchase_time,


    case when min(view_time) is not null and min(cart_time) is not null and min(view_time) < min(cart_time)
    then datediff('seconds', min(view_time), min(cart_time))
    end sec_fView_to_fCart,

    case when min(view_time) is not null and min(purchase_time) is not null and min(purchase_time) > min(view_time)
    then datediff('seconds', min(view_time), min(purchase_time))
    end sec_fView_to_fPurchase,

    case when min(cart_time) is not null and min(purchase_time) is not null and min(cart_time) < min(purchase_time)
    then datediff ('seconds',min(cart_time), min(purchase_time))
    end sec_fCart_to_fPurchase,


---creating a straight order view->cart->purchase------

  case
        when min(view_time) is not null and min(cart_time) is not null and min(purchase_time)is not null
        and min(view_time) < min(cart_time) and min(cart_time) < min(purchase_time)
        then 1 else 0
        end usual_Path_purchase,

---- user path --------

 case 
    when max(saw_purchased) = 1 and max(saw_cart) =1 and max(saw_view) = 1
    then 'view->cart->purchase'

    when max(saw_cart) = 1 and max(saw_purchased) = 1 and max(saw_view) = 0
    then 'cart->purchase'

    when max(saw_cart) = 1 and max(saw_view) = 1 
    then 'view->cart'

    when max(saw_view) = 1 and max(saw_cart) = 0 
    then 'view_only'

    when max(saw_cart) =1 and max(saw_view) = 0
    then 'cart_only'

    when max(saw_purchased) = 1 and max(saw_cart) = 0
    then 'purchase_without_cart'

else 'other'
end user_path ,

sum(case when event_type = 'purchase' then coalesce(price,0) else 0 end) session_revenue,

count(distinct product_id) distinct_prod_inSession

from processed.events_processed

group by user_id, user_session ;
        

        
        



In [ ]:
-----had to recreate the table as the few coulumns were not queried properly-------

create
or replace table processed.events_processed as 

with enriching as (
    select
        *,
        min(
            case
                when event_type = 'view' then event_time
            end
        ) over(partition by user_id, user_session, product_id) view_time,
        min(
            case
                when event_type = 'cart' then event_time
            end
        ) over (partition by user_id, user_session, product_id) cart_time,
        min(
            case
                when event_type = 'remove_from_cart' then event_time
            end
        ) over (partition by user_id, user_session, product_id) removeCart_time,
        min(
            case
                when event_type = 'purchase' then event_time
            end
        ) over (partition by user_id, user_session, product_id) purchase_time
    from
        raw.events_cleaned
)
select
    user_id,
    user_session,
    event_time,
    event_type,
    product_id,
    category_id,
    price,
    event_date,
    saw_view,
    saw_cart,
    saw_removeCart,
    saw_purchased,
    view_time,
    cart_time,
    removeCart_time,
    purchase_time,
    case
        when view_time is not null
        and cart_time is not null
        and cart_time > view_time then datediff('seconds', view_time, cart_time)
    end sec_view_toCart,
    case
        when view_time is not null
        and purchase_time is not null
        and purchase_time > view_time then datediff('seconds', view_time, purchase_time)
    end sec_view_toPurchase,
    case
        when cart_time is not null
        and purchase_time is not null
        and purchase_time > cart_time then datediff('seconds', cart_time, purchase_time)
    end sec_cart_toPurchase,
    case
        when cart_time is not null
        and removeCart_time is not null
        and removeCart_time > cart_time then datediff('seconds', cart_time, removeCart_time)
    end sec_cart_toRemoveCart
from
    enriching ;

In [ ]:
-------the final table for the analytics schema at user level for LTV type analysis-----

create or replace table analysis.events_userLvl as 

with  each_user_kpi as (

select user_id,

        ----acitvity and revenue-----
        count(distinct user_session) total_sessions,
        sum(case when event_type = 'purchase' then coalesce(price,0)else 0 end) lifetime_revenue,

        -----distinct user engagement counts----
        count(distinct case when saw_view = 1 then product_id end ) distinct_prod_alltime_viewed,
        count(distinct case when saw_purchased = 1 then product_id end ) distinct_prod_alltime_purchased,

        -----customer activity recency---------

        min(event_time) first_event_time,
        max(event_time) last_event_time,

        ---------check the last recent purchase of the customer to gauge the dormant customers----
        max(case when event_type = 'purchase' then event_time end ) latest_order_date,


        ------ to get he first of the funnels , to know the behaviour of the new customer------

        min(view_time) first_view_time,
        min(cart_time) first_cart_time,
        min(purchase_time) first_purchase_time

    from processed.events_processed

group by user_id 

)

select 
    user_id,
    total_sessions,
    lifetime_revenue,

    distinct_prod_alltime_viewed,
    distinct_prod_alltime_purchased,

    first_event_time,
    last_event_time,
    latest_order_date,
    case when latest_order_date is not null then 'yes' else 'no' end is_converted,
    first_view_time,
    first_cart_time,
    first_purchase_time,

    case 
    when first_view_time is not null and first_purchase_time is not null 
    and first_view_time < first_purchase_time
    then datediff('second',first_view_time, first_purchase_time) 
    end secs_firstView_to_firstPurchase,

    case 
    when first_cart_time is not null and first_purchase_time is not null
    and first_cart_time < first_purchase_time
    then datediff('second',first_cart_time, first_purchase_time)
    end  secs_firstCart_to_firstPurchase

    from each_user_kpi
    ;
    
    

In [ ]:
select *
from analysis.events_userlvl
where last_event_time != latest_order_date
limit 20

In [ ]:
copy into @~/events.Prodlvl.csv    ----
from analysis.events_prodlevel
file_format = (
type = 'CSV'
field_delimiter = ','
field_optionally_enclosed_by = '"'
null_if = ('\\N','NULL', '')
)
header = true
overwrite = true
single = true
max_file_size = 1104857600;  

In [ ]:
remove @~/events.Prodlvl.csv

In [ ]:
copy into @~/events.Prodlvl.csv    ----
from analysis.events_prodlevel
file_format = (
type = 'CSV'
field_delimiter = ','
field_optionally_enclosed_by = '"'
null_if = ('\\N','NULL', '')
)
header = true
overwrite = true
single = true
max_file_size = 1104857600;  

In [ ]:
list @~/events.Prodlvl.csv


In [ ]:
copy into @~/events.SessionLvl.csv
from analysis.events_sessionlevel
file_format = (
type = 'CSV'
field_delimiter = ','
field_optionally_enclosed_by = '"'
null_if = ('//N','NULL','')
)
header = true
overwrite = true
single = true
max_file_size = 1104857600 ;

In [ ]:
list @~/events.SessionLvl.csv

In [ ]:
copy into @~/events.UserLvl.csv
from analysis.events_userlvl
file_format = (
type = 'CSV'
field_delimiter = ','
field_optionally_enclosed_by ='"'
null_if =('//N','NULL','')
)
header = true
overwrite = true
single = true
max_file_size = 1104857600 ;


In [ ]:
create or replace stage my_export_stage;

In [ ]:
copy into @my_export_stage/events.ProdLvl.csv    ----
from analysis.events_prodlevel
file_format = (
type = 'CSV'
field_delimiter = ','
field_optionally_enclosed_by = '"'
null_if = ('\\N','NULL', '')
)
header = true
overwrite = true
single = true
max_file_size = 1104857600; 

In [ ]:
remove @~/events.Prodlvl.csv

In [ ]:
copy into @my_export_stage/events.SessionLvl.csv
from analysis.events_sessionlevel
file_format = (
type = 'CSV'
field_delimiter = ','
field_optionally_enclosed_by = '"'
null_if = ('//N','NULL','')
)
header = true
overwrite = true
single = true
max_file_size = 1104857600 ;

In [ ]:
remove @~/events.SessionLvl.csv

In [ ]:
copy into @my_export_stage/events.UserLvl.csv
from analysis.events_userlvl
file_format = (
type = 'CSV'
field_delimiter = ','
field_optionally_enclosed_by ='"'
null_if =('//N','NULL','')
)
header = true
overwrite = true
single = true
max_file_size = 1104857600 ;

In [ ]:
remove @my_export_stage/events.UserLvl.csv